In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from modellens.core.lens import ModelLens

model = GPT2LMHeadModel.from_pretrained("gpt2-medium", attn_implementation="eager")
model.config.output_attentions = True
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")

lens = ModelLens(model)
lens.adapter.set_tokenizer(tokenizer)

print(lens)
print(f"Backend: {lens.adapter.type_of_adapter}")
print(f"Layers: {len(lens.layer_names())}")

/opt/anaconda3/envs/modellens/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 292/292 [00:00<00:00, 8482.67it/s]


ModelLens(backend=huggingface, hooks=0, params=354,823,168)
Backend: huggingface
Layers: 319


## Logit Lens

In [2]:
from modellens.analysis.logit_lens import run_logit_lens, decode_logit_lens

prompt = "The capital of France is"
tokens = tokenizer(prompt, return_tensors="pt")
top_k = 5

lens.attach_all()
results = run_logit_lens(lens, tokens, top_k=top_k)
decoded = decode_logit_lens(results, tokenizer=tokenizer)

print(f"Prompt: '{prompt}'\n")
for layer_name, predictions in decoded.items():
    tokens_str = ", ".join(f"{tok!r} ({prob:.3f})" for tok, prob in predictions)
    print(f"{layer_name:40s} -> {tokens_str}")

Prompt: 'The capital of France is'

transformer.wte                          -> ' is' (0.000), ' has' (0.000), ' isn' (0.000), ' does' (0.000), ' appears' (0.000)
transformer.wpe                          -> 'anwhile' (0.000), 'SPONSORED' (0.000), 'ippi' (0.000), 'ック' (0.000), 'ña' (0.000)
transformer.drop                         -> ' is' (0.000), ' helicop' (0.000), ' has' (0.000), ' unden' (0.000), ' does' (0.000)
transformer.h.0.ln_1                     -> ' is' (0.000), ' has' (0.000), ' was' (0.000), ' does' (0.000), ' isn' (0.000)
transformer.h.0.attn.c_proj              -> ' offensive' (0.000), ' interior' (0.000), ' Magazine' (0.000), 'initions' (0.000), ' premiere' (0.000)
transformer.h.0.attn.resid_dropout       -> ' offensive' (0.000), ' interior' (0.000), ' Magazine' (0.000), 'initions' (0.000), ' premiere' (0.000)
transformer.h.0.attn                     -> ' offensive' (0.000), ' interior' (0.000), ' Magazine' (0.000), 'initions' (0.000), ' premiere' (0.000)
transformer.h.

## Attention Analysis

In [3]:
from modellens.analysis.attention import run_attention_analysis, head_summary

attn_results = run_attention_analysis(lens, prompt)

print(f"Number of attention layers: {attn_results['num_layers']}\n")
for name, data in list(attn_results["attention_maps"].items())[:5]:
    print(f"{name}:")
    print(f"  Heads: {data['num_heads']}, Seq length: {data['seq_length']}")
    print(f"  Weights shape: {data['weights'].shape}")

# %% Head Summary — focused vs diffuse heads
summary = head_summary(attn_results)
for name, data in list(summary.items())[:5]:
    print(f"\n{name}:")
    print(f"  Entropy:       {[f'{e:.2f}' for e in data['entropy']]}")
    print(f"  Max attention: {[f'{a:.2f}' for a in data['max_attention']]}")

Number of attention layers: 24

transformer.h.0.attn:
  Heads: 16, Seq length: 5
  Weights shape: torch.Size([1, 16, 5, 5])
transformer.h.1.attn:
  Heads: 16, Seq length: 5
  Weights shape: torch.Size([1, 16, 5, 5])
transformer.h.2.attn:
  Heads: 16, Seq length: 5
  Weights shape: torch.Size([1, 16, 5, 5])
transformer.h.3.attn:
  Heads: 16, Seq length: 5
  Weights shape: torch.Size([1, 16, 5, 5])
transformer.h.4.attn:
  Heads: 16, Seq length: 5
  Weights shape: torch.Size([1, 16, 5, 5])

transformer.h.0.attn:
  Entropy:       ['0.88', '0.90', '0.47', '0.71', '0.84', '0.80', '0.68', '0.72', '0.85', '0.88', '0.75', '0.92', '0.73', '0.80', '0.84', '0.85']
  Max attention: ['0.56', '0.56', '0.79', '0.72', '0.63', '0.66', '0.70', '0.74', '0.61', '0.60', '0.70', '0.54', '0.72', '0.68', '0.63', '0.64']

transformer.h.1.attn:
  Entropy:       ['0.65', '0.48', '0.55', '0.78', '0.60', '0.58', '0.87', '0.56', '0.47', '0.72', '0.24', '0.58', '0.58', '0.58', '0.77', '0.92']
  Max attention: ['0.77'

In [4]:
tokens_list = tokenizer.tokenize(prompt)
print(f"Tokens: {tokens_list}\n")

weights = attn_results["attention_maps"]["transformer.h.4.attn"]["weights"]
head_13 = weights[0, 13]

for i, token in enumerate(tokens_list):
    attn_values = ", ".join(f"{tokens_list[j]} ({head_13[i][j]:.3f})" for j in range(len(tokens_list)))
    print(f"{token:10s} attends to: {attn_values}")

Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']

The        attends to: The (1.000), Ġcapital (0.000), Ġof (0.000), ĠFrance (0.000), Ġis (0.000)
Ġcapital   attends to: The (0.998), Ġcapital (0.002), Ġof (0.000), ĠFrance (0.000), Ġis (0.000)
Ġof        attends to: The (0.000), Ġcapital (1.000), Ġof (0.000), ĠFrance (0.000), Ġis (0.000)
ĠFrance    attends to: The (0.000), Ġcapital (0.000), Ġof (0.999), ĠFrance (0.000), Ġis (0.000)
Ġis        attends to: The (0.000), Ġcapital (0.000), Ġof (0.000), ĠFrance (0.999), Ġis (0.000)


## Residual Stream Analysis

In [5]:
from modellens.analysis.residual_stream import run_residual_analysis, identify_critical_layers

block_layers = [f"transformer.h.{i}" for i in range(24)]
lens.attach_layers(block_layers)
residual_results = run_residual_analysis(lens, tokens, layer_names=block_layers)

print("Per-layer residual stream contributions:\n")
for name, data in residual_results["contributions"].items():
    bar = "█" * int(data["relative_contribution"] * 50)
    print(f"{name:20s} | rel: {data['relative_contribution']:.3f} | cos: {data['cosine_similarity']:.3f} | {bar}")

critical = identify_critical_layers(residual_results, threshold=0.05)
print(f"\nCritical layers: {critical}")

Per-layer residual stream contributions:

transformer.h.1      | rel: 0.192 | cos: 0.985 | █████████
transformer.h.2      | rel: 0.387 | cos: 0.985 | ███████████████████
transformer.h.3      | rel: 0.829 | cos: 0.901 | █████████████████████████████████████████
transformer.h.4      | rel: 0.056 | cos: 0.992 | ██
transformer.h.5      | rel: 0.058 | cos: 0.992 | ██
transformer.h.6      | rel: 0.037 | cos: 0.994 | █
transformer.h.7      | rel: 0.034 | cos: 0.991 | █
transformer.h.8      | rel: 0.026 | cos: 0.993 | █
transformer.h.9      | rel: 0.023 | cos: 0.993 | █
transformer.h.10     | rel: 0.025 | cos: 0.992 | █
transformer.h.11     | rel: 0.025 | cos: 0.992 | █
transformer.h.12     | rel: 0.026 | cos: 0.992 | █
transformer.h.13     | rel: 0.028 | cos: 0.993 | █
transformer.h.14     | rel: 0.037 | cos: 0.989 | █
transformer.h.15     | rel: 0.037 | cos: 0.990 | █
transformer.h.16     | rel: 0.045 | cos: 0.989 | ██
transformer.h.17     | rel: 0.042 | cos: 0.992 | ██
transformer.h.18     

## Activation Patching

In [6]:
from modellens.analysis.activation_patching import run_activation_patching

# Clear any leftover hooks
for name, module in model.named_modules():
    module._forward_hooks.clear()

clean = tokenizer("The capital of France is", return_tensors="pt")
corrupted = tokenizer("The MX of France is", return_tensors="pt")

def france_metric(output):
    if hasattr(output, "logits"):
        output = output.logits
    probs = torch.softmax(output[:, -1, :], dim=-1)
    paris_id = tokenizer.encode(" Paris")[0]
    return probs[:, paris_id].mean().item()

patch_results = run_activation_patching(
    lens, clean, corrupted, metric_fn=france_metric
)

print(f"Clean P(Paris):     {patch_results['clean_metric']:.4f}")
print(f"Corrupted P(Paris): {patch_results['corrupted_metric']:.4f}")
print(f"Total effect:       {patch_results['total_effect']:.4f}\n")

for layer, data in patch_results["patch_effects"].items():
    bar = "█" * int(abs(data["normalized_effect"]) * 20)
    sign = "+" if data["normalized_effect"] >= 0 else "-"
    print(f"{layer:35s} | P(Paris): {data['patched_metric']:.4f} | effect: {sign}{abs(data['normalized_effect']):.3f} | {bar}")


Clean P(Paris):     0.1741
Corrupted P(Paris): 0.0001
Total effect:       -0.1739

transformer.h.0.attn                | P(Paris): 0.2112 | effect: -0.214 | ████
transformer.h.0.mlp                 | P(Paris): 0.0001 | effect: +1.000 | ███████████████████
transformer.h.1.attn                | P(Paris): 0.1870 | effect: -0.075 | █
transformer.h.1.mlp                 | P(Paris): 0.2045 | effect: -0.175 | ███
transformer.h.2.attn                | P(Paris): 0.1831 | effect: -0.052 | █
transformer.h.2.mlp                 | P(Paris): 0.1828 | effect: -0.050 | █
transformer.h.3.attn                | P(Paris): 0.1384 | effect: +0.205 | ████
transformer.h.3.mlp                 | P(Paris): 0.1416 | effect: +0.187 | ███
transformer.h.4.attn                | P(Paris): 0.1398 | effect: +0.197 | ███
transformer.h.4.mlp                 | P(Paris): 0.1881 | effect: -0.081 | █
transformer.h.5.attn                | P(Paris): 0.1648 | effect: +0.053 | █
transformer.h.5.mlp                 | P(Paris): 0.0

## Embeddings Analysis

In [7]:
from modellens.analysis.embeddings import run_embeddings_analysis

clean = tokenizer("The capital of France is", return_tensors="pt")
embed_results = run_embeddings_analysis(lens, clean)
tokens_list = tokenizer.tokenize("The capital of France is")

print(f"Tokens: {tokens_list}")
print(f"Embedding dim: {embed_results['embed_dim']}")
print(f"Sequence length: {embed_results['seq_length']}")
print(f"\nPer-token norms:")
for tok, norm in zip(tokens_list, embed_results['norms'][0].tolist()):
    print(f"  {tok:10s} -> {norm:.4f}")

print(f"\nCosine similarity matrix:")
sim = embed_results["similarity_matrix"].numpy().round(3)
print(f"{'':12s} {'  '.join(f'{t:>8s}' for t in tokens_list)}")
for i, tok in enumerate(tokens_list):
    row = "  ".join(f"{sim[i][j]:8.3f}" for j in range(len(tokens_list)))
    print(f"{tok:12s} {row}")

Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
Embedding dim: 1024
Sequence length: 5

Per-token norms:
  The        -> 2.6075
  Ġcapital   -> 3.2428
  Ġof        -> 2.1251
  ĠFrance    -> 3.0892
  Ġis        -> 2.1739

Cosine similarity matrix:
                  The  Ġcapital       Ġof   ĠFrance       Ġis
The             1.000     0.312     0.492     0.421     0.533
Ġcapital        0.312     1.000     0.261     0.263     0.283
Ġof             0.492     0.261     1.000     0.265     0.569
ĠFrance         0.421     0.263     0.265     1.000     0.304
Ġis             0.533     0.283     0.569     0.304     1.000
